### 个人信息保护中间件 PIIMiddleware

`PIIMiddleware` 用于检测并处理对话中的**个人身份信息**（PII），可在用户输入、模型输出、工具返回这几个环节扫描敏感数据并做脱敏。

**内置可检测的 PII 类型**：

- `email`：邮箱（正则）
- `credit_card`：信用卡号（正则 + Luhn 校验）
- `ip`：IP 地址（正则 + 标准库校验）
- `mac_address`：MAC 地址
- `url`：URL（带协议或裸域名）

**四种处理策略**：

| 策略 | 效果 | 示例（邮箱） | 适用场景 |
| --- | --- | --- | --- |
| `block` | 检测到即抛 `PIIDetectionError` | 抛异常 | 完全不允许出现 PII |
| `redact`（默认） | 替换为占位符 | `[REDACTED_EMAIL]` | 合规、日志脱敏 |
| `mask` | 只保留少量信息 | `alice@****.com` | 客服等需要人工可读 |
| `hash` | 确定性哈希（可关联） | `<email_hash:ff8d9819>` | 分析、调试 |

#### 构造参数

```python
PIIMiddleware(
    pii_type,
    *,
    strategy="redact",
    detector=None,
    apply_to_input=True,
    apply_to_output=False,
    apply_to_tool_results=False,
)
```

- `pii_type`：要检测的类型。内置类型见上；也可以是自定义名称（此时必须提供 `detector`）
- `strategy`：`block` / `redact` / `mask` / `hash`
- `detector`：自定义检测器
  - 传 `str`：作为正则表达式
  - 传 `Callable[[str], list[PIIMatch]]`：自定义检测函数
  - 传 `None`：使用内置检测器
- `apply_to_input`：是否处理**用户输入**（默认 `True`）
- `apply_to_output`：是否处理**模型输出**（默认 `False`）
- `apply_to_tool_results`：是否处理**工具返回结果**（默认 `False`）

> 一个中间件只负责一种 `pii_type`；要处理多种类型就配置多个中间件。

In [1]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import PIIDetectionError, PIIMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

load_dotenv(override=True)

model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


#### 案例 1：四种策略效果对比（不调用模型）

In [2]:
# 直接使用底层检测器 + apply_strategy，快速看清各策略对同一段文本的处理结果
from langchain.agents.middleware._redaction import (
    apply_strategy,
    detect_credit_card,
    detect_email,
    detect_ip,
    detect_mac_address,
    detect_url,
)

# ---- email ----
email_text = "联系我：alice@example.com"
email_matches = detect_email(email_text)
print("email     redact:", apply_strategy(email_text, email_matches, "redact"))
print("email     mask  :", apply_strategy(email_text, email_matches, "mask"))
print("email     hash  :", apply_strategy(email_text, email_matches, "hash"))

# ---- credit_card（会做 Luhn 校验，4111... 可通过）----
cc_text = "卡号 4111 1111 1111 1111"
print("credit    mask  :", apply_strategy(cc_text, detect_credit_card(cc_text), "mask"))

# ---- ip ----
ip_text = "服务器 192.168.1.100"
print("ip        mask  :", apply_strategy(ip_text, detect_ip(ip_text), "mask"))

# ---- mac ----
mac_text = "设备 00:1A:2B:3C:4D:5E"
print("mac       mask  :", apply_strategy(mac_text, detect_mac_address(mac_text), "mask"))

# ---- url ----
url_text = "访问 https://secret.example.com/path?token=abc"
print("url       redact:", apply_strategy(url_text, detect_url(url_text), "redact"))

# ---- block：不返回文本，直接抛异常 ----
try:
    apply_strategy(email_text, email_matches, "block")
except PIIDetectionError as e:
    print("email     block : 抛出", type(e).__name__, "-", e)


email     redact: 联系我：[REDACTED_EMAIL]
email     mask  : 联系我：alice@****.com
email     hash  : 联系我：<email_hash:ff8d9819>
credit    mask  : 卡号 **** **** **** 1111
ip        mask  : 服务器 *.*.*.100
mac       mask  : 设备 **:**:**:**:**:5E
url       redact: 访问 [REDACTED_URL]
email     block : 抛出 PIIDetectionError - Detected 1 instance(s) of email in text content


#### 案例 2：redact（默认）—— 脱敏用户输入

In [3]:
# 默认 apply_to_input=True：模型调用前，最后一条用户消息会被脱敏
agent_redact = create_agent(
    model=model,
    middleware=[
        PIIMiddleware("email", strategy="redact"),
    ],
)

result = agent_redact.invoke(
    {"messages": [{"role": "user", "content": "请记录我的邮箱 alice@example.com，谢谢"}]}
)

# 注意：state 里的用户消息已经被替换成了脱敏后的内容
print("脱敏后的用户消息：", result["messages"][0].content)


脱敏后的用户消息： 请记录我的邮箱 [REDACTED_EMAIL]，谢谢


#### 案例 3：block —— 检测到 PII 直接抛异常

In [4]:
# strategy="block" 时，一旦在用户输入里发现邮箱就会抛 PIIDetectionError
agent_block = create_agent(
    model=model,
    middleware=[
        PIIMiddleware("email", strategy="block"),
    ],
)

try:
    agent_block.invoke({"messages": [{"role": "user", "content": "我的邮箱是 alice@example.com"}]})
    print("未检测到 PII（不符合预期）")
except PIIDetectionError as e:
    print("已按预期抛出：", type(e).__name__)
    print("错误信息：", e)


已按预期抛出： PIIDetectionError
错误信息： Detected 1 instance(s) of email in text content


#### 案例 4：apply_to_output —— 脱敏模型输出

In [5]:
# apply_to_input=False 让模型能看到原始邮箱（用于复述），
# apply_to_output=True 则在模型回复后对 AI 消息脱敏
agent_output = create_agent(
    model=model,
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=False,   # 不处理输入
            apply_to_output=True,   # 处理模型输出
        )
    ],
)

result = agent_output.invoke(
    {"messages": [{"role": "user", "content": "请原样重复这句话：我的邮箱是 alice@example.com"}]}
)

print("模型回复（已被脱敏）：", result["messages"][-1].content)


模型回复（已被脱敏）： 我的邮箱是 [REDACTED_EMAIL]


#### 案例 5：apply_to_tool_results —— 脱敏工具返回

In [6]:
@tool
def get_customer() -> str:
    """查询客户信息（这里故意返回含 PII 的内容）。"""
    return "客户邮箱 bob@example.com，IP 10.0.0.1"


# apply_to_tool_results=True：工具执行后、再次调用模型前，对 ToolMessage 脱敏
agent_tool = create_agent(
    model=model,
    tools=[get_customer],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_tool_results=True),
    ],
)

result = agent_tool.invoke({"messages": [{"role": "user", "content": "帮我查一下客户信息"}]})

for m in result["messages"]:
    if type(m).__name__ == "ToolMessage":
        print("工具返回（已被脱敏）：", m.content)


工具返回（已被脱敏）： 客户邮箱 [REDACTED_EMAIL]，IP 10.0.0.1


#### 案例 6：自定义 PII 类型（正则 / 可调用 detector）

In [7]:
# 非内置类型必须提供 detector
# 1) 用正则字符串：识别 sk- 开头的 API Key
agent_regex = create_agent(
    model=model,
    middleware=[
        PIIMiddleware("api_key", detector=r"sk-[A-Za-z0-9]{12}", strategy="redact"),
    ],
)

result = agent_regex.invoke(
    {"messages": [{"role": "user", "content": "我的 key 是 sk-abc123456789，请记住"}]}
)
print("正则 detector：", result["messages"][0].content)

# 2) 用可调用 detector：完全自定义匹配逻辑（这里识别 6 位数字验证码）
def detect_otp(content: str):
    import re

    return [
        {"type": "otp", "value": m.group(), "start": m.start(), "end": m.end()}
        for m in re.finditer(r"\b\d{6}\b", content)
    ]


agent_callable = create_agent(
    model=model,
    middleware=[
        PIIMiddleware("otp", detector=detect_otp, strategy="mask"),
    ],
)

result = agent_callable.invoke(
    {"messages": [{"role": "user", "content": "我的验证码是 654321"}]}
)
print("可调用 detector：", result["messages"][0].content)


正则 detector： 我的 key 是 [REDACTED_API_KEY]，请记住
可调用 detector： 我的验证码是 ****4321


#### 案例 7：多种 PII 类型 + 不同策略

In [ ]:
# 一个中间件只管一种类型，多种类型就叠加多个中间件
agent_multi = create_agent(
    model=model,
    middleware=[
        PIIMiddleware("email", strategy="redact"),          # 邮箱 -> [REDACTED_EMAIL]
        PIIMiddleware("credit_card", strategy="mask"),       # 信用卡 -> **** **** **** 1111
        PIIMiddleware("ip", strategy="hash"),                # IP -> <ip_hash:xxxx>
    ],
)

result = agent_multi.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "我的邮箱 alice@example.com，卡号 4111 1111 1111 1111，服务器 192.168.1.100",
            }
        ]
    }
)
print("脱敏后的用户消息：", result["messages"][0].content)


#### 要点回顾

1. 一个 `PIIMiddleware` 只处理一种 `pii_type`，多种类型叠加多个实例。
2. 四种策略：`block`（抛异常）/ `redact`（占位符）/ `mask`（部分保留）/ `hash`（确定性哈希）。
3. 三个作用范围：`apply_to_input`（默认开）、`apply_to_output`、`apply_to_tool_results`。
4. 内置类型之外的 PII，用 `pii_type` + `detector`（正则或回调）自定义。
5. `credit_card` 会做 Luhn 校验，`ip` 会做标准库校验，能减少误报。